In [11]:
import pandas as pd
import numpy as np

# =========================
# LOAD DATASETS
# =========================

sleep_df = pd.read_csv("Dataset/Sleep_Health_and_Lifestyle_Dataset.csv")
student_df = pd.read_csv("Dataset/student_productivity_dataset.csv")

# =========================
# SELECT USEFUL COLUMNS
# =========================

sleep_df = sleep_df[
    [
        "Sleep Duration",
        "Stress Level",
        "Physical Activity Level"
    ]
]

student_df = student_df[
    [
        "study_hours_per_day",
        "sleep_hours",
        "phone_usage_hours",
        "social_media_hours",
        "youtube_hours",
        "gaming_hours",
        "breaks_per_day",
        "coffee_intake_mg",
        "exercise_minutes",
        "assignments_completed",
        "attendance_percentage",
        "stress_level",
        "focus_score"
    ]
]

# =========================
# RENAME SLEEP DATA COLUMNS
# =========================

sleep_df.columns = [
    "sleep_hours_real",
    "stress_level_real",
    "exercise_real"
]

# =========================
# EXPAND SMALL DATASET
# =========================

sleep_sampled = sleep_df.sample(
    n=len(student_df),
    replace=True,
    random_state=42
).reset_index(drop=True)

student_df = student_df.reset_index(drop=True)

# =========================
# MERGE DATASETS
# =========================

merged_df = pd.concat(
    [student_df, sleep_sampled],
    axis=1
)

# =========================
# CREATE FINAL FEATURES
# =========================

merged_df["sleep_hours_final"] = (
    merged_df["sleep_hours"] +
    merged_df["sleep_hours_real"]
) / 2

merged_df["stress_final"] = (
    merged_df["stress_level"] +
    merged_df["stress_level_real"]
) / 2

merged_df["exercise_final"] = (
    merged_df["exercise_minutes"] +
    merged_df["exercise_real"]
) / 2

# =========================
# DROP UNUSED COLUMNS
# =========================

merged_df.drop(
    columns=[
        "sleep_hours",
        "sleep_hours_real",
        "stress_level",
        "stress_level_real",
        "exercise_minutes",
        "exercise_real"
    ],
    inplace=True
)

# =========================
# CREATE PRODUCTIVITY SCORE
# =========================

# Base productivity
merged_df["productivity_score"] = 50

# -------------------------
# POSITIVE FACTORS
# -------------------------

merged_df["productivity_score"] += (
    merged_df["study_hours_per_day"] * 2.5
)

merged_df["productivity_score"] += (
    merged_df["focus_score"] * 0.30
)

merged_df["productivity_score"] += (
    merged_df["attendance_percentage"] * 0.10
)

merged_df["productivity_score"] += (
    merged_df["assignments_completed"] * 1.2
)

merged_df["productivity_score"] += (
    merged_df["exercise_final"] * 0.05
)

# -------------------------
# SLEEP EFFECT
# -------------------------

# Best productivity around 7–8 hours sleep

sleep_penalty = abs(
    merged_df["sleep_hours_final"] - 7.5
) * 4

merged_df["productivity_score"] -= sleep_penalty

# -------------------------
# NEGATIVE FACTORS
# -------------------------

merged_df["productivity_score"] -= (
    merged_df["phone_usage_hours"] * 1.5
)

merged_df["productivity_score"] -= (
    merged_df["social_media_hours"] * 1.8
)

merged_df["productivity_score"] -= (
    merged_df["gaming_hours"] * 1.2
)

merged_df["productivity_score"] -= (
    merged_df["stress_final"] * 2.5
)

# -------------------------
# RANDOM HUMAN VARIATION
# -------------------------

noise = np.random.normal(
    0,
    8,
    len(merged_df)
)

merged_df["productivity_score"] += noise

# -------------------------
# CLIP SCORE RANGE
# -------------------------

merged_df["productivity_score"] = (
    merged_df["productivity_score"]
    .clip(0, 100)
)

# =========================
# CREATE PRODUCTIVITY LABEL
# =========================

def productivity_label(score):

    if score < 50:
        return "Low"

    elif score < 72:
        return "Medium"

    else:
        return "High"

merged_df["productivity_label"] = (
    merged_df["productivity_score"]
    .apply(productivity_label)
)

# =========================
# CHECK CLASS DISTRIBUTION
# =========================

print("\nClass Distribution:\n")

print(
    merged_df["productivity_label"]
    .value_counts(normalize=True) * 100
)

# =========================
# DATASET INFO
# =========================

print("\nDataset Shape:\n")
print(merged_df.shape)

print("\nProductivity Score Stats:\n")
print(
    merged_df["productivity_score"]
    .describe()
)

# =========================
# SAVE FINAL DATASET
# =========================

merged_df.to_csv(
    "final_productivity_dataset.csv",
    index=False
)

print("\nSaved Successfully ✅")


Class Distribution:

productivity_label
Medium    47.465
High      35.490
Low       17.045
Name: proportion, dtype: float64

Dataset Shape:

(20000, 15)

Productivity Score Stats:

count    20000.000000
mean        65.740877
std         16.231878
min          7.214106
25%         54.644952
50%         65.774942
75%         76.992693
max        100.000000
Name: productivity_score, dtype: float64

Saved Successfully ✅


In [12]:
final = pd.read_csv("final_productivity_dataset.csv")

In [17]:
final = final.rename(columns = {"sleep_hours_final":"sleep_hours", "stress_final":"stress_level", "exercise_final": "exercise_minutes"})

In [18]:
final.head(10)

,study_hours_per_day,phone_usage_hours,social_media_hours,youtube_hours,gaming_hours,breaks_per_day,coffee_intake_mg,assignments_completed,attendance_percentage,focus_score,sleep_hours,stress_level,exercise_minutes,productivity_score,productivity_label
0,4.35,3.38,2.73,1.83,5.26,6,347,2,57.21,57,5.415,7.0,85.5,37.413481,Low
1,6.14,5.48,1.51,3.13,1.73,13,403,10,91.27,49,7.390,6.5,51.5,64.572195,Medium
2,4.98,4.83,3.63,0.18,4.71,1,419,8,63.14,38,4.680,5.0,96.0,33.910829,Low
3,3.19,10.06,3.95,5.75,2.52,9,178,18,40.51,50,5.340,6.0,35.0,44.919211,Low
4,7.67,3.02,1.59,5.46,5.65,8,436,7,45.53,41,6.155,7.0,67.5,71.967496,Medium
5,7.18,4.02,3.74,1.42,0.16,10,392,3,47.58,70,5.110,7.0,28.5,55.546590,Medium
6,9.06,11.45,5.99,2.20,4.44,14,87,15,43.50,35,7.030,7.0,51.5,49.237119,Low
7,6.37,3.31,1.37,4.36,5.13,2,152,17,75.22,59,6.030,5.0,81.5,86.568347,High
8,4.19,9.66,2.87,0.10,3.38,13,460,11,44.79,39,6.035,3.5,51.0,45.703169,Low
9,7.28,2.13,0.81,1.35,2.55,7,416,6,79.15,73,8.680,7.5,98.5,93.508059,High
